In [1]:
import shap
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import spearmanr
from pathlib import Path
import random
import numpy as np
import torch
import pandas as pd

from models import TwoHeadMLPNet, SingleOutMLPNet
from shap_helpers import (compute_shap_3out, 
                          _ensure_2d_shap, 
                          mean_abs_shap, 
                          topk_table, 
                          global_table,
                          save_fig, 
                          cosine_spearman)

### MLP

In [2]:
shap_mlp = np.load("./Results_MLP/shap_values/shap_values_MLP_multioutput.npz", allow_pickle=True)

OUTDIR = Path("./FiguresPaper")
SHAP_DIR = OUTDIR / "shap_values"
FIG_DIR = SHAP_DIR / "figs"
TAB_DIR = SHAP_DIR / "tables"

TOPK = 5
mtag = "MLP_multi"

In [3]:
sv_list = [shap_mlp['shap_age'], shap_mlp['shap_MetSCORE'], shap_mlp['shap_sex']]
X_ex = shap_mlp["X_ex"]
feature_names = shap_mlp["feature_names"].tolist()

In [4]:
out_names = ["age", "MetSCORE", "sex"]
mlp_multi_meanabs = {}

for j, outn in enumerate(out_names):
    sv2 = _ensure_2d_shap(sv_list[j], tag=f"{mtag}:{outn}")
    ma = mean_abs_shap(sv2, tag=f"{mtag}:{outn}")
    mlp_multi_meanabs[outn] = ma

    df_top = topk_table(ma, feature_names, TOPK, tag=f"{mtag}_{outn}", tabdir=TAB_DIR)
    _ = global_table(ma, feature_names, tag=f"{mtag}_{outn}", tabdir=TAB_DIR)

    top_feats = df_top["feature"].tolist()
    top_idx = [feature_names.index(f) for f in top_feats]
    sv_top = sv2[:, top_idx]
    X_ex_top = X_ex[:, top_idx]

    plt.figure()
    shap.summary_plot(sv_top, features=X_ex_top, feature_names=top_feats, show=False)
    plt.gca().set_title(f"MLP multi-output: ({outn})", loc="center",pad=15)
    save_fig(FIG_DIR / f"SHAP_{mtag}_top{TOPK}_{outn}")

### Transformer

In [5]:
shap_tr = np.load("./Results_TR/shap_values/shap_values_TR_multioutput.npz", allow_pickle=True)

TOPK = 5
mtag = "TR_multi"

In [6]:
sv_list = [shap_tr['shap_age'], shap_tr['shap_MetSCORE'], shap_tr['shap_sex']]
X_ex = shap_tr["X_ex"]
feature_names = shap_tr["feature_names"].tolist()

In [7]:
out_names = ["age", "MetSCORE", "sex"]
tr_multi_meanabs = {}

for j, outn in enumerate(out_names):
    sv2 = _ensure_2d_shap(sv_list[j], tag=f"{mtag}:{outn}")
    ma = mean_abs_shap(sv2, tag=f"{mtag}:{outn}")
    tr_multi_meanabs[outn] = ma

    df_top = topk_table(ma, feature_names, TOPK, tag=f"{mtag}_{outn}", tabdir=TAB_DIR)
    _ = global_table(ma, feature_names, tag=f"{mtag}_{outn}", tabdir=TAB_DIR)

    top_feats = df_top["feature"].tolist()
    top_idx = [feature_names.index(f) for f in top_feats]
    sv_top = sv2[:, top_idx]
    X_ex_top = X_ex[:, top_idx]

    plt.figure()
    shap.summary_plot(sv_top, features=X_ex_top, feature_names=top_feats, show=False, 
                      cmap="plasma")
    plt.gca().set_title(f"Transformer multi-output: ({outn})", loc="center",pad=15)
    save_fig(FIG_DIR / f"SHAP_{mtag}_top{TOPK}_{outn}")

In [18]:
np.mean(sv_top, axis=0)

array([ 0.06312233,  0.00184568, -0.01963652,  0.01696238,  0.00138734])